# Solar Filament Segmentation 2026 — Approach A: high-resolution CNN ensemble + learned PQ post-processing

**Settings → Accelerator → GPU T4 x2, Internet ON**, then *Save & Run All* (about 4.5 h).

**Why this should beat the first U-Net.** The first notebook lost points in three places. This one attacks each:

| Weakness of the first U-Net | What this notebook does |
|---|---|
| Half resolution (1024 px): filaments are only a few pixels wide, so mask edges (SQ) suffer | Works at **1536 px** with 768 px crops, and trims final masks at 2048 px along the probability map |
| Raw grey image in, one model, one split, ~4.5k steps | **Physics channels** (limb-darkening-corrected intensity, local contrast, dark-ridge filter) into an ImageNet-pretrained **EfficientNetV2-S U-Net**; **4-fold** ensemble, one fold per GPU in parallel (full single-GPU speed each) |
| Fixed thresholds; every component submitted | All post-processing tuned on **out-of-fold** predictions of all 707 images: hysteresis thresholds, a **learned fragment-merge** model, and a **learned component scorer** that keeps a filament only if its expected matched IoU beats the PQ break-even |

**Comparable numbers.** Besides the out-of-fold PQ on all 707 images, it reports PQ on exactly the validation
images of the first U-Net notebook, so you can compare directly.

In [ ]:
import os, sys, json, time, glob, subprocess, threading, shutil, importlib, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2, torch
warnings.filterwarnings("ignore")

DRY = os.environ.get("SFC_DRYRUN") == "1"          # tiny CPU smoke test only; leave unset on Kaggle
T0 = time.time()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("SFC_WORK", "./work")).resolve()
WORK.mkdir(parents=True, exist_ok=True); os.chdir(WORK); sys.path.insert(0, str(WORK))
TEMP = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path(os.environ.get("SFC_TEMP", "/tmp/sfc_temp"))
TEMP.mkdir(parents=True, exist_ok=True)
N_GPU = torch.cuda.device_count()

CFG = dict(
    res=int(os.environ.get("SFC_DRY_RES", 256)) if DRY else 1536,   # working resolution (0.75 x native)
    crop=int(os.environ.get("SFC_DRY_RES", 256)) // 2 if DRY else 768,   # training crop
    encoder="resnet18" if DRY else "tf_efficientnetv2_s.in21k_ft_in1k",
    encoder_fallbacks=[] if DRY else ["resnet50.a1_in1k", "resnet34.a1_in1k"],
    batch=4 if DRY else 8,                         # per GPU; halved automatically on out-of-memory
    lr=3e-4, ema=0.999, seed=42, log_every=5 if DRY else 250,
    workers=0 if DRY else max((os.cpu_count() or 4) // max(N_GPU, 1), 1),
    max_steps=int(os.environ.get("SFC_DRY_STEPS", 25)) if DRY else 10 ** 9,
)
N_FOLDS = 2 if DRY else 4
FOLD_HOURS = 1.0 if DRY else 1.75                  # training time per fold; 2 folds run at once on 2 GPUs
print("GPUs:", N_GPU, [torch.cuda.get_device_name(g) for g in range(N_GPU)], "| work", WORK, "| temp", TEMP)
if not DRY and N_GPU < 2:
    print("!!! fewer than 2 GPUs: folds will run one after another (about twice as long). Choose GPU T4 x2.")

In [ ]:
%%writefile sfc_common.py
"""sfc_common -- approach-independent utilities shared by both v3 notebooks (identical copy in each).

Data registry, physics-inspired input channels, memory-mapped caches, the competition metric (PQ), instance helpers,
the baseline validation split (for like-for-like comparison with the first U-Net submission) and submission export."""
from __future__ import annotations
import os, json, math, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
from scipy import ndimage as ndi
from pycocotools import mask as M

FULL = 2048
STATIONS = ("Bh", "Ch", "Lh", "Mh", "Th", "Uh")


# ============================================================================ data
def find_root():
    cands = [os.environ.get("SFC_ROOT"), "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"]
    for c in cands:
        if c and (Path(c) / "train").exists():
            return Path(c)
    hits = sorted(Path("/kaggle/input").rglob("*Annotations*train*.json")) if Path("/kaggle/input").exists() else []
    assert hits, "competition data not found under /kaggle/input - attach the competition data to the notebook"
    return hits[0].parent.parent


class Data:
    """Annotation registry. `stems` has the same order as the first (baseline) notebook, so split indices match."""

    def __init__(self, root):
        root = Path(root)
        ann = sorted((root / "train").glob("*Annotations*train*.json"))[0]
        d = json.load(open(ann))
        self.anns, self.recs = {}, {}
        for a in d["annotations"]:
            self.anns.setdefault(a["image_id"], []).append(a)
        for im in d["images"]:
            self.recs.setdefault(im["file_name"].rsplit(".", 1)[0], []).append(im["id"])
        ext = (".jpeg", ".jpg", ".png")
        self.img_path = {p.stem: p for p in (root / "train" / "train_images").glob("*") if p.suffix.lower() in ext}
        self.stems = sorted(s for s in self.recs if s in self.img_path)
        self.test_paths = sorted(p for p in (root / "test" / "test_images").glob("*") if p.suffix.lower() in ext)

    def rles(self, rid, size):
        """One RLE per annotated filament of one annotator record (polygons are in 2048-px coordinates)."""
        s = size / FULL
        return [M.merge(M.frPyObjects([(np.asarray(p, float) * s).tolist() for p in a["segmentation"]], size, size))
                for a in self.anns.get(rid, [])]

    def labels(self, rid, size):
        lab = np.zeros((size, size), np.uint8)
        for k, r in enumerate(self.rles(rid, size), 1):
            lab[M.decode(r) > 0] = min(k, 255)
        return lab

    def gt(self, idx, size):
        return {r: self.rles(r, size) for i in idx for r in self.recs[self.stems[i]]}


def baseline_val_idx(n, seed=42, frac=0.15):
    """The exact held-out images of the first U-Net notebook (same seed / fraction / ordering)."""
    perm = np.random.default_rng(seed).permutation(n)
    return np.sort(perm[:max(int(round(frac * n)), 2)])


def make_folds(stems, k):
    """Observation-level folds, dealt round-robin after sorting by (station, timestamp): every fold sees every station
    and the whole time range; consecutive frames of the same filament fall in different folds."""
    order = sorted(range(len(stems)), key=lambda i: (stems[i][-2:], stems[i][:-2]))
    f = np.zeros(len(stems), int)
    for j, i in enumerate(order):
        f[i] = j % k
    return f


# ============================================================================ physics-inspired channels
def disk_fit(img8):
    b = cv2.GaussianBlur(img8, (0, 0), 3)
    _, m = cv2.threshold(b, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    h, w = img8.shape
    if n < 2:
        return w / 2, h / 2, 0.45 * min(h, w)
    comp = ndi.binary_fill_holes(lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA]))
    ys, xs = np.nonzero(comp)
    cx, cy, R = xs.mean(), ys.mean(), math.sqrt(comp.sum() / math.pi)
    if not (0.25 * w < R < 0.62 * w):                       # implausible fit: never let one odd frame break a run
        return w / 2, h / 2, 0.45 * min(h, w)
    return cx, cy, R


def preprocess(path, res):
    """-> chans (3,res,res) f16 [flat-field, local contrast, dark-ridge], gray (res,res) f16 (disk-standardised),
    valid (res,res) bool, disk (cx, cy, R).

    flat-field : intensity / radial median profile  (removes limb darkening, keeps local dark structures)
    contrast   : (flat - local mean) / local std    (normalised convolution restricted to the disk)
    dark ridge : max over scales of sigma^2 * largest Hessian eigenvalue (> 0 across dark, elongated structures)"""
    img8 = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    img8 = cv2.resize(img8, (res, res), interpolation=cv2.INTER_AREA)
    img = img8.astype(np.float32)
    cx, cy, R = disk_fit(img8)
    yy, xx = np.mgrid[0:res, 0:res].astype(np.float32)
    rr = np.hypot(xx - cx, yy - cy) / R
    valid = rr < 0.985
    nb = 100
    bins = np.clip((rr[valid] * nb).astype(int), 0, nb - 1)
    order = np.argsort(bins, kind="stable")
    b_s, v_s = bins[order], img[valid][order]
    bounds = np.searchsorted(b_s, np.arange(nb + 1))
    prof = np.array([np.median(v_s[bounds[i]:bounds[i + 1]]) if bounds[i + 1] - bounds[i] >= 20 else np.nan
                     for i in range(nb)], np.float32)
    cen = (np.arange(nb) + 0.5) / nb
    ok = ~np.isnan(prof)
    prof = np.interp(cen, cen[ok], prof[ok]) if ok.any() else np.full(nb, max(img[valid].mean(), 1.0))
    flat = np.where(valid, img / np.maximum(np.interp(rr, cen, prof), 1.0), 1.0).astype(np.float32)
    w = valid.astype(np.float32)
    s = res / 1024.0

    def nblur(a, sig):
        return cv2.GaussianBlur(a * w, (0, 0), sig) / np.maximum(cv2.GaussianBlur(w, (0, 0), sig), 1e-3)

    mu = nblur(flat, 16 * s)
    dlt = (flat - mu) * w
    z = dlt / np.maximum(np.sqrt(np.maximum(nblur(dlt ** 2, 32 * s), 1e-8)), 0.02)
    ridge = np.zeros_like(flat)
    for sig in (1.5 * s, 3 * s, 6 * s):
        g = cv2.GaussianBlur(flat, (0, 0), sig)
        gxx = cv2.Sobel(g, cv2.CV_32F, 2, 0, ksize=3); gyy = cv2.Sobel(g, cv2.CV_32F, 0, 2, ksize=3)
        gxy = cv2.Sobel(g, cv2.CV_32F, 1, 1, ksize=3)
        l1 = 0.5 * (gxx + gyy) + np.sqrt(0.25 * (gxx - gyy) ** 2 + gxy ** 2)
        ridge = np.maximum(ridge, sig ** 2 * np.maximum(l1, 0))
    ridge *= w
    ridge = np.log1p(ridge / (4 * np.median(ridge[valid]) + 1e-6)) if valid.any() else ridge
    chans = np.stack([np.clip((flat - 1) / 0.25, -3, 3) * w, np.clip(z, -4, 4) * w, np.clip(ridge, 0, 4) * w])
    gray = np.clip((img - img[valid].mean()) / (img[valid].std() + 1e-6), -5, 5) * w if valid.any() else img * 0
    return chans.astype(np.float16), gray.astype(np.float16), valid, (cx, cy, R)


# ============================================================================ memory-mapped caches
def build_cache(data, out_dir, res, test=False, n_jobs=None, log=print):
    """Train cache: chans, gray, valid, soft (annotator-mean mask, x255), labels (one uint8 label map per annotator
    record) + rec_start offsets. Test cache: chans, gray, valid. Resumes if a complete cache already exists."""
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    man_p = out / "manifest.json"
    items = [p.stem for p in data.test_paths] if test else list(data.stems)
    if man_p.exists():
        man = json.load(open(man_p))
        if man.get("res") == res and man.get("items") == items:
            log(f"cache {out} already complete ({len(items)} images)")
            return man
    paths = data.test_paths if test else [data.img_path[s] for s in items]
    N = len(items)
    mm = lambda name, dt, shape: np.lib.format.open_memmap(out / f"{name}.npy", "w+", dt, shape)
    chans, gray, valid = mm("chans", np.float16, (N, 3, res, res)), mm("gray", np.float16, (N, res, res)), mm("valid", np.uint8, (N, res, res))
    disk = np.zeros((N, 3), np.float32)
    if not test:
        rec_start = np.cumsum([0] + [len(data.recs[s]) for s in items])
        soft, labels = mm("soft", np.uint8, (N, res, res)), mm("labels", np.uint8, (int(rec_start[-1]), res, res))

    def one(i):
        c, g, v, dk = preprocess(paths[i], res)
        chans[i], gray[i], valid[i], disk[i] = c, g, v, dk
        if not test:
            acc = np.zeros((res, res), np.float32)
            for j, rid in enumerate(data.recs[items[i]]):
                lab = data.labels(rid, res)
                labels[rec_start[i] + j] = lab
                acc += lab > 0
            soft[i] = np.round(acc / len(data.recs[items[i]]) * 255).astype(np.uint8)
        return i

    t = time.time()
    with ThreadPoolExecutor(n_jobs or os.cpu_count() or 1) as ex:
        for k, _ in enumerate(ex.map(one, range(N))):
            if k % 100 == 0 or k == N - 1:
                log(f"  cached {k + 1}/{N} ({time.time() - t:.0f}s)")
    for a in (chans, gray, valid) + (() if test else (soft, labels)):
        a.flush()
    np.save(out / "disk.npy", disk)
    if not test:
        np.save(out / "rec_start.npy", rec_start)
    man = {"res": res, "items": items, "test": test}
    json.dump(man, open(man_p, "w"))
    return man


def open_cache(out_dir):
    out = Path(out_dir)
    c = {"manifest": json.load(open(out / "manifest.json"))}
    for name in ("chans", "gray", "valid", "soft", "labels"):
        if (out / f"{name}.npy").exists():
            c[name] = np.load(out / f"{name}.npy", mmap_mode="r")
    c["disk"] = np.load(out / "disk.npy")
    if (out / "rec_start.npy").exists():
        c["rec_start"] = np.load(out / "rec_start.npy")
    return c


# ============================================================================ metric
def pq(gt, preds):
    """Competition metric: every annotator record is scored separately against the image's predictions, then pooled.
    hit = IoU > 0.5 for each (gt, prediction) cell; PQ = sum IoU(hits) / (TP + FP/2 + FN/2)."""
    S = TP = FP = FN = 0.0
    for rid, gts in gt.items():
        pr = preds.get(rid.split("-", 1)[1], [])
        if not gts:
            FP += len(pr); continue
        if not pr:
            FN += len(gts); continue
        iou = np.asarray(M.iou(pr, gts, [0] * len(gts))).T
        hit = iou > 0.5
        S += iou[hit].sum(); TP += hit.sum(); FP += (hit.sum(0) == 0).sum(); FN += (hit.sum(1) == 0).sum()
    den = TP + 0.5 * FP + 0.5 * FN
    return dict(pq=float(S / den) if den else 0.0, sq=float(S / TP) if TP else 0.0, dq=float(TP / den) if den else 0.0,
                tp=int(TP), fp=int(FP), fn=int(FN))


def matched_iou(lab, gt_rles_by_rec):
    """Per component of `lab`: mean over annotator records of IoU * 1[IoU > 0.5] (its expected contribution to PQ's
    numerator). With disjoint predictions at most one GT per record can exceed 0.5."""
    n = int(lab.max())
    if n == 0:
        return np.zeros(0)
    crl = [M.encode(np.asfortranarray((lab == k).astype(np.uint8))) for k in range(1, n + 1)]
    acc = np.zeros(n)
    for rid, gts in gt_rles_by_rec.items():
        if gts:
            iou = np.asarray(M.iou(crl, gts, [0] * len(gts)))       # (n, G)
            acc += (iou * (iou > 0.5)).sum(1)
    return acc / max(len(gt_rles_by_rec), 1)


# ============================================================================ instance helpers
def instances_cc(prob, valid, t_hi, t_lo, min_area, close=1):
    """Hysteresis threshold -> small closing -> connected components (disjoint by construction)."""
    ext = ((prob >= t_lo) & valid).astype(np.uint8)
    n, lab = cv2.connectedComponents(ext, connectivity=8)
    if n <= 1:
        return np.zeros(prob.shape, np.int32)
    seeds = np.unique(lab[(prob >= t_hi) & valid]); seeds = seeds[seeds > 0]
    m = np.isin(lab, seeds)
    if close > 0 and m.any():
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close + 1, 2 * close + 1))
        m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_CLOSE, k).astype(bool) & valid
    n, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    keep = np.zeros(n, np.int32)
    ids = np.flatnonzero(st[:, cv2.CC_STAT_AREA] >= min_area); ids = ids[ids > 0]
    keep[ids] = np.arange(1, len(ids) + 1)
    return keep[lab]


def relabel(lab):
    u, inv = np.unique(lab, return_inverse=True)
    out = inv.reshape(lab.shape).astype(np.int32)
    return out if u[0] == 0 else out + 1


def to_rles(lab, keep, prob=None, thr=None, size=FULL):
    """RLEs of the kept labels at `size`: nearest-neighbour label up-sampling, trimmed by the bilinearly up-sampled
    probability so boundaries follow the map rather than pixel blocks. Disjoint by construction."""
    if size != lab.shape[0]:
        ix = (np.arange(size) * lab.shape[0]) // size
        lab = lab[ix][:, ix]
        if prob is not None and thr:
            lab = lab * (cv2.resize(prob.astype(np.float32), (size, size), interpolation=cv2.INTER_LINEAR) >= thr)
    out = []
    for k in keep:
        m = lab == k
        if m.any():
            out.append(M.encode(np.asfortranarray(m.astype(np.uint8))))
    return out


def label_rles(lab):
    return {int(k): M.encode(np.asfortranarray((lab == k).astype(np.uint8))) for k in np.unique(lab) if k > 0}


def select(score, thr, min_keep=1):
    """Labels (1-based) with score >= thr; never fewer than min_keep (every training record has >= 1 filament)."""
    score = np.asarray(score)
    keep = np.flatnonzero(score >= thr) + 1
    if len(keep) < min_keep and len(score):
        keep = np.argsort(-score)[:min_keep] + 1
    return keep.tolist()


def rot_flip(a, k, f):
    a = np.rot90(a, k, axes=(-2, -1))
    return np.ascontiguousarray(np.flip(a, -1) if f else a)


# ============================================================================ submission
def write_submission(preds, path):
    rows = [{"filament_id": f"{s}_{k}", "segmentation_rle": r["counts"].decode("ascii") if isinstance(r["counts"], bytes) else r["counts"]}
            for s in sorted(preds) for k, r in enumerate(preds[s], 1)]
    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    df.to_csv(path, index=False)
    return df


def validate_submission(df, test_stems):
    img = df.filament_id.str.rsplit("_", n=1).str[0]
    assert df.filament_id.is_unique, "duplicate filament_id"
    unknown = set(img) - set(test_stems)
    assert not unknown, f"rows for unknown images {sorted(unknown)[:3]}"
    for s, g in df.groupby(img):
        L = M.decode([{"size": [FULL, FULL], "counts": c.encode()} for c in g.segmentation_rle])
        assert L.reshape(-1, L.shape[2]).sum(0).min() > 0, f"{s}: empty mask"
        assert L.sum(2).max() <= 1, f"{s}: overlapping masks"
    missing = sorted(set(test_stems) - set(img))
    return {"rows": len(df), "images": int(img.nunique()), "per_image": round(len(df) / max(len(test_stems), 1), 2),
            "images_without_rows": missing}

In [ ]:
%%writefile a_lib.py
"""a_lib -- Approach A: high-resolution CNN U-Net ensemble + learned, PQ-optimal post-processing.

Training runs as `python a_lib.py train --fold k ...`, one process per GPU (two folds train in parallel on 2xT4,
each at full single-GPU speed -- no DataParallel overhead)."""
from __future__ import annotations
import os, sys, json, math, time, argparse, copy
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from skimage import measure
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from pycocotools import mask as M
import sfc_common as S


# ============================================================================ model
def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class UNetA(nn.Module):
    """timm encoder (strides 2..32) + U-Net decoder + a full-resolution block joined with the raw input channels."""

    def __init__(self, encoder, dec=(256, 128, 64, 32), pretrained=False):
        super().__init__()
        import timm
        self.enc = timm.create_model(encoder, features_only=True, pretrained=pretrained, in_chans=3)
        ch = self.enc.feature_info.channels()
        self.dec = nn.ModuleList()
        cin = ch[-1]
        for skip, wo in zip(reversed(ch[:-1]), dec):
            self.dec.append(block(cin + skip, wo)); cin = wo
        self.last = block(cin + 3, 24)
        self.head = nn.Sequential(nn.Conv2d(24, 24, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(24, 1, 1))

    def forward(self, x):
        h, w = x.shape[-2:]
        p = (-h) % 32, (-w) % 32
        xp = F.pad(x, (0, p[1], 0, p[0]))
        fs = self.enc(xp)
        y = fs[-1]
        for blk, s in zip(self.dec, reversed(fs[:-1])):
            y = blk(torch.cat([F.interpolate(y, size=s.shape[-2:], mode="bilinear", align_corners=False), s], 1))
        y = self.last(torch.cat([F.interpolate(y, size=xp.shape[-2:], mode="bilinear", align_corners=False), xp], 1))
        return self.head(y)[..., :h, :w]


def build_model(cfg, init_path=None):
    m = UNetA(cfg["encoder"])
    if init_path and os.path.exists(init_path):
        m.enc.load_state_dict(torch.load(init_path, map_location="cpu", weights_only=True))
    return m


def save_pretrained_encoder(cfg, out_path, log=print):
    """Parent process: fetch ImageNet weights once (hub, or an attached file) and save them for the fold processes."""
    import timm, glob
    for enc in [cfg["encoder"]] + list(cfg.get("encoder_fallbacks", [])):
        try:
            e = timm.create_model(enc, features_only=True, pretrained=True, in_chans=3)
            torch.save(e.state_dict(), out_path); return enc, "timm hub"
        except Exception as ex:
            log(f"  {enc}: hub download failed ({type(ex).__name__})")
        base = enc.split(".")[0]
        for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
            if base in p.lower() and p.endswith((".safetensors", ".pth", ".bin")):
                try:
                    e = timm.create_model(enc, features_only=True, pretrained=True, in_chans=3, pretrained_cfg_overlay=dict(file=p))
                    torch.save(e.state_dict(), out_path); return enc, p
                except Exception:
                    pass
    return cfg["encoder"], None


# ============================================================================ training data
class CropDS(torch.utils.data.Dataset):
    def __init__(self, cache_dir, idx, crop, length, seed):
        self.cache_dir, self.idx, self.crop, self.length, self.seed = cache_dir, np.asarray(idx), crop, length, seed
        self.c = None

    def _open(self):
        self.c = S.open_cache(self.cache_dir)
        soft = self.c["soft"]
        self.fg = {}
        for i in self.idx:                                       # filament pixel coordinates (stride 8) for crop centring
            ys, xs = np.nonzero(np.asarray(soft[i, ::8, ::8]) > 127)
            self.fg[int(i)] = np.stack([ys * 8, xs * 8], 1)

    def __len__(self):
        return self.length

    def __getitem__(self, k):
        if self.c is None:
            self._open()
        r = np.random
        R, C = self.c["soft"].shape[-1], self.crop
        i = int(r.choice(self.idx)); fg = self.fg[i]
        if len(fg) and r.rand() < 0.7:
            cy, cx = fg[r.randint(len(fg))]
            y0 = int(np.clip(cy - C // 2 + r.randint(-C // 3, C // 3 + 1), 0, R - C))
            x0 = int(np.clip(cx - C // 2 + r.randint(-C // 3, C // 3 + 1), 0, R - C))
        else:
            y0, x0 = r.randint(R - C + 1), r.randint(R - C + 1)
        sl = (slice(y0, y0 + C), slice(x0, x0 + C))
        x = np.asarray(self.c["chans"][i][(slice(None),) + sl], np.float32)
        soft = np.asarray(self.c["soft"][i][sl], np.float32) / 255
        valid = np.asarray(self.c["valid"][i][sl], np.float32)
        k_, f_ = r.randint(4), r.rand() < 0.5
        x, soft, valid = S.rot_flip(x, k_, f_), S.rot_flip(soft, k_, f_), S.rot_flip(valid, k_, f_)
        x = (x * r.uniform(0.85, 1.15, (3, 1, 1)) + r.normal(0, 0.05, (3, 1, 1))) * valid
        return torch.from_numpy(x.astype(np.float32)), torch.from_numpy(soft[None]), torch.from_numpy(valid[None])


def loss_fn(logit, soft, valid):
    bce = (F.binary_cross_entropy_with_logits(logit, soft, reduction="none") * valid).sum() / valid.sum().clamp_min(1)
    p = torch.sigmoid(logit) * valid
    dice = 1 - (2 * (p * soft).sum() + 1) / (p.sum() + (soft * valid).sum() + 1)
    return bce + dice


def train_main(a):
    cfg = json.load(open(os.path.join(a.work, "cfg_a.json")))
    torch.manual_seed(cfg["seed"] + a.fold); np.random.seed(cfg["seed"] + a.fold)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    folds = np.array(json.load(open(os.path.join(a.work, "folds_a.json"))))
    tr = np.flatnonzero(folds != a.fold) if a.fold >= 0 else np.arange(len(folds))
    model = build_model(cfg, os.path.join(a.work, "enc_init_a.pt")).to(dev)
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters():
        p.requires_grad_(False)
    enc_p = [p for n, p in model.named_parameters() if n.startswith("enc.")]
    dec_p = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    opt = torch.optim.AdamW([{"params": enc_p, "base": cfg["lr"] * 0.5}, {"params": dec_p, "base": cfg["lr"]}], weight_decay=1e-4)
    amp = dev.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    bs = cfg["batch"]
    while True:                                                   # largest batch that fits (OOM -> halve)
        try:
            ds = CropDS(a.cache, tr, cfg["crop"], 10 ** 8, cfg["seed"] + a.fold)
            dl = torch.utils.data.DataLoader(ds, batch_size=bs, num_workers=cfg["workers"], drop_last=True,
                                             persistent_workers=cfg["workers"] > 0, pin_memory=amp,
                                             worker_init_fn=lambda w: np.random.seed((cfg["seed"] * 1000 + a.fold * 100 + w + int(time.time())) % 2 ** 31))
            it = iter(dl)
            x, soft, valid = (t.to(dev) for t in next(it))
            with torch.autocast(device_type=dev.type, enabled=amp):
                out = model(x)
            loss_fn(out.float(), soft, valid).backward(); opt.zero_grad(set_to_none=True)
            break
        except torch.cuda.OutOfMemoryError:
            del dl; torch.cuda.empty_cache(); bs //= 2
            print(f"OOM -> batch {bs}", flush=True)
            assert bs >= 1
    budget, t0, step, hist = a.hours * 3600, time.time(), 0, []
    print(f"fold {a.fold}: {len(tr)} train images | batch {bs} | crop {cfg['crop']} | {a.hours:.2f} h budget | {dev}", flush=True)
    model.train()
    for x, soft, valid in it:
        frac = (time.time() - t0) / budget
        if frac >= 1 or step >= cfg.get("max_steps", 10 ** 9):
            break
        for g in opt.param_groups:
            g["lr"] = g["base"] * min(1.0, (step + 1) / 300) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * frac)))
        x, soft, valid = x.to(dev, non_blocking=True), soft.to(dev, non_blocking=True), valid.to(dev, non_blocking=True)
        with torch.autocast(device_type=dev.type, enabled=amp):
            out = model(x)
        loss = loss_fn(out.float(), soft, valid)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            d = min(cfg["ema"], (1 + step) / (10 + step))
            for pe, pm in zip(ema.state_dict().values(), model.state_dict().values()):
                if pe.dtype.is_floating_point:
                    pe.mul_(d).add_(pm.detach(), alpha=1 - d)
                else:
                    pe.copy_(pm)
        hist.append(loss.item()); step += 1
        if step % cfg["log_every"] == 0:
            el = time.time() - t0
            print(f"fold {a.fold} | step {step} | {el / 60:.0f} min | {step * bs / el:.1f} crops/s | loss {np.mean(hist[-cfg['log_every']:]):.4f}", flush=True)
    torch.save(ema.state_dict(), os.path.join(a.work, f"a_fold{a.fold}.pt"))
    json.dump({"fold": a.fold, "steps": step, "batch": bs, "minutes": (time.time() - t0) / 60,
               "final_loss": float(np.mean(hist[-200:])) if hist else None}, open(os.path.join(a.work, f"a_fold{a.fold}.json"), "w"))
    print(f"fold {a.fold} done: {step} steps in {(time.time() - t0) / 60:.0f} min", flush=True)


# ============================================================================ inference
def load_models(cfg, paths, dev):
    ms = []
    for p in paths:
        m = build_model(cfg).to(dev)
        m.load_state_dict(torch.load(p, map_location=dev, weights_only=True)); m.eval(); ms.append(m)
    return ms


@torch.no_grad()
def predict_prob(models, chans, valid, dev, tta=("id", "hflip")):
    x = torch.from_numpy(np.asarray(chans, np.float32) * np.asarray(valid, np.float32))[None].to(dev)
    acc, n = 0, 0
    for m in models:
        for t in tta:
            xi = torch.flip(x, (-1,)) if t == "hflip" else (torch.flip(x, (-2,)) if t == "vflip" else x)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                p = torch.sigmoid(m(xi).float())
            acc = acc + (torch.flip(p, (-1,)) if t == "hflip" else (torch.flip(p, (-2,)) if t == "vflip" else p)); n += 1
    return (acc / n)[0, 0].cpu().numpy() * np.asarray(valid, np.float32)


# ============================================================================ learned fragment merging
PAIR_FEATURES = ["gap", "gap_rel", "area_min", "area_max", "w_min", "w_max", "p_min", "p_max", "elong_min", "elong_max",
                 "align_min", "align_max", "dir_cos", "line_p_mean", "line_p_min", "line_c0", "line_c1", "line_c2",
                 "r_mid", "n_comp", "cent_dist"]


def _local_dir(lab, label, y, x, rad):
    H, W = lab.shape
    yy, xx = np.nonzero(lab[max(y - rad, 0):min(y + rad + 1, H), max(x - rad, 0):min(x + rad + 1, W)] == label)
    if len(yy) < 5:
        return None
    c = np.stack([yy, xx], 1).astype(np.float64); c -= c.mean(0)
    return np.linalg.eigh(c.T @ c)[1][:, 1]


def pair_table(lab, prob, chans, rr, max_gap):
    """One row per pair of components whose closest contour points are within max_gap px."""
    n = int(lab.max())
    cols = ["a", "b", "ya", "xa", "yb", "xb", "bw"] + PAIR_FEATURES
    if n < 2:
        return pd.DataFrame(columns=cols)
    s = lab.shape[0] / 1024.0
    props = measure.regionprops(lab)
    pts, own, info = [], [], {}
    for p in props:
        y0, x0, y1, x1 = p.bbox
        sub = (lab[y0:y1, x0:x1] == p.label).astype(np.uint8)
        c = np.concatenate(cv2.findContours(sub, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0])[:, 0, :]
        if len(c) > 600:
            c = c[:: int(np.ceil(len(c) / 600))]
        pts.append(np.c_[c[:, 1] + y0, c[:, 0] + x0]); own.append(np.full(len(c), p.label))
        dt = cv2.distanceTransform(sub, cv2.DIST_L2, 3)
        info[p.label] = dict(area=p.area, w=2.0 * float(dt[sub > 0].mean()), p=float(prob[y0:y1, x0:x1][sub > 0].mean()),
                             el=max(p.major_axis_length, 1e-3) / max(p.minor_axis_length, 1e-3), cen=p.centroid)
    P, O = np.concatenate(pts), np.concatenate(own)
    ij = cKDTree(P).query_pairs(r=max_gap, output_type="ndarray")
    if len(ij) == 0:
        return pd.DataFrame(columns=cols)
    oa, ob = O[ij[:, 0]], O[ij[:, 1]]
    k = oa != ob; ij, oa, ob = ij[k], oa[k], ob[k]
    if len(ij) == 0:
        return pd.DataFrame(columns=cols)
    sw = oa > ob; ij[sw] = ij[sw][:, ::-1]; oa, ob = np.minimum(oa, ob), np.maximum(oa, ob)
    d = np.linalg.norm(P[ij[:, 0]] - P[ij[:, 1]], axis=1)
    key = oa.astype(np.int64) * (n + 1) + ob
    order = np.lexsort((d, key))
    first = order[np.r_[True, key[order][1:] != key[order][:-1]]]
    c0, c1, c2 = (np.asarray(chans[q], np.float32) for q in range(3))
    rows, rad = [], int(round(12 * s))
    for q in first:
        a_, b_ = int(oa[q]), int(ob[q])
        (ya, xa), (yb, xb) = P[ij[q, 0]].astype(int), P[ij[q, 1]].astype(int)
        gap = float(d[q]); ia, ib = info[a_], info[b_]
        m = max(int(np.ceil(gap)), 1) + 1
        ly = np.clip(np.round(np.linspace(ya, yb, m)).astype(int), 0, lab.shape[0] - 1)
        lx = np.clip(np.round(np.linspace(xa, xb, m)).astype(int), 0, lab.shape[1] - 1)
        lp = prob[ly, lx]
        u = np.array([yb - ya, xb - xa], float); un = np.linalg.norm(u)
        da, db = _local_dir(lab, a_, ya, xa, rad), _local_dir(lab, b_, yb, xb, rad)
        if un < 1e-6:
            al_a = al_b = 1.0
        else:
            u /= un
            al_a = abs(float(da @ u)) if da is not None else 0.5
            al_b = abs(float(db @ u)) if db is not None else 0.5
        rows.append(dict(a=a_, b=b_, ya=ya, xa=xa, yb=yb, xb=xb, bw=max(min(ia["w"], ib["w"]), 1.0),
                         gap=gap / s, gap_rel=gap / max(min(ia["w"], ib["w"]), 1.0),
                         area_min=min(ia["area"], ib["area"]) / s ** 2, area_max=max(ia["area"], ib["area"]) / s ** 2,
                         w_min=min(ia["w"], ib["w"]) / s, w_max=max(ia["w"], ib["w"]) / s,
                         p_min=min(ia["p"], ib["p"]), p_max=max(ia["p"], ib["p"]),
                         elong_min=min(ia["el"], ib["el"]), elong_max=max(ia["el"], ib["el"]),
                         align_min=min(al_a, al_b), align_max=max(al_a, al_b),
                         dir_cos=abs(float(da @ db)) if (da is not None and db is not None) else 0.5,
                         line_p_mean=float(lp.mean()), line_p_min=float(lp.min()),
                         line_c0=float(c0[ly, lx].mean()), line_c1=float(c1[ly, lx].mean()), line_c2=float(c2[ly, lx].mean()),
                         r_mid=float(rr[(ya + yb) // 2, (xa + xb) // 2]), n_comp=n,
                         cent_dist=float(np.hypot(ia["cen"][0] - ib["cen"][0], ia["cen"][1] - ib["cen"][1])) / s))
    return pd.DataFrame(rows)[cols]


def pair_targets(lab, pairs, rec_labels, min_frac=0.3):
    """Fraction of annotator records in which both components lie mostly inside the SAME annotated filament."""
    if len(pairs) == 0:
        return np.zeros(0)
    n = int(lab.max()); labf = lab.ravel().astype(np.int64)
    a, b = pairs["a"].to_numpy(int), pairs["b"].to_numpy(int)
    acc = np.zeros(len(pairs))
    for gl in rec_labels:
        G = int(gl.max())
        if G == 0:
            continue
        h = np.bincount(labf * (G + 1) + gl.ravel().astype(np.int64), minlength=(n + 1) * (G + 1)).reshape(n + 1, G + 1)
        best = h[:, 1:].argmax(1) + 1
        frac = h[np.arange(n + 1), best] / np.maximum(h.sum(1), 1)
        assign = np.where(frac >= min_frac, best, -np.arange(n + 1) - 1)
        acc += assign[a] == assign[b]
    return acc / max(len(rec_labels), 1)


def apply_merges(lab, pairs, scores, thr, valid):
    sel = np.flatnonzero(np.asarray(scores) >= thr) if len(pairs) else []
    if len(sel) == 0:
        return lab
    n = int(lab.max()); parent = np.arange(n + 1)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x

    out = lab.astype(np.int32, copy=True); H, W = out.shape
    for q in sel:
        r = pairs.iloc[q]; a_, b_ = int(r.a), int(r.b)
        ra, rb = find(a_), find(b_)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
        t = int(np.clip(round(r.bw), 1, 15)); ya, xa, yb, xb = int(r.ya), int(r.xa), int(r.yb), int(r.xb)
        y0, y1, x0, x1 = max(min(ya, yb) - t, 0), min(max(ya, yb) + t + 1, H), max(min(xa, xb) - t, 0), min(max(xa, xb) + t + 1, W)
        patch = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.line(patch, (xa - x0, ya - y0), (xb - x0, yb - y0), 1, t)
        sub = out[y0:y1, x0:x1]
        sub[(patch > 0) & (sub == 0) & valid[y0:y1, x0:x1]] = a_       # bridge only background pixels (stays disjoint)
    roots = np.array([find(i) for i in range(n + 1)])
    return S.relabel(roots[out])


# ============================================================================ learned component scorer
COMP_FEATURES = ["area", "length", "width", "elong", "solidity", "p_mean", "p_max", "p_p90", "p_min", "c0_mean", "c0_min",
                 "c1_mean", "c2_mean", "c2_max", "ring_c0", "contrast", "r_cen", "r_min", "n_comp", "area_rank", "nn_dist",
                 "img_p_mean", "station"]


def comp_table(lab, prob, chans, rr, valid, station):
    n = int(lab.max())
    if n == 0:
        return pd.DataFrame(columns=COMP_FEATURES)
    s = lab.shape[0] / 1024.0
    idx = np.arange(1, n + 1)
    c0, c1, c2 = (np.asarray(chans[q], np.float32) for q in range(3))
    f = dict(area=ndi.sum(np.ones_like(prob), lab, idx) / s ** 2, p_mean=ndi.mean(prob, lab, idx), p_max=ndi.maximum(prob, lab, idx),
             p_min=ndi.minimum(prob, lab, idx), c0_mean=ndi.mean(c0, lab, idx), c0_min=ndi.minimum(c0, lab, idx),
             c1_mean=ndi.mean(c1, lab, idx), c2_mean=ndi.mean(c2, lab, idx), c2_max=ndi.maximum(c2, lab, idx),
             r_min=ndi.minimum(rr, lab, idx))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * max(int(6 * s), 1) + 1,) * 2)
    rows = []
    props = measure.regionprops(lab)
    cents = np.array([p.centroid for p in props])
    for p in props:
        y0, x0, y1, x1 = p.bbox; pad = int(8 * s) + 1
        ya, xa, yb, xb = max(y0 - pad, 0), max(x0 - pad, 0), min(y1 + pad, lab.shape[0]), min(x1 + pad, lab.shape[1])
        sub = (lab[ya:yb, xa:xb] == p.label).astype(np.uint8)
        ring = cv2.dilate(sub, k).astype(bool) & ~sub.astype(bool) & (lab[ya:yb, xa:xb] == 0) & valid[ya:yb, xa:xb]
        ring_c0 = float(c0[ya:yb, xa:xb][ring].mean()) if ring.any() else float(c0[ya:yb, xa:xb][sub > 0].mean())
        cnts = cv2.findContours(sub, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        hull = cv2.contourArea(cv2.convexHull(np.concatenate(cnts))) if cnts else 0.0
        (_, _), (w_, h_), _ = cv2.minAreaRect(np.concatenate(cnts).astype(np.float32)) if cnts else ((0, 0), (1, 1), 0)
        rows.append(dict(length=max(w_, h_) / s, width=p.area / max(max(w_, h_), 1.0) / s,
                         elong=max(p.major_axis_length, 1e-3) / max(p.minor_axis_length, 1e-3),
                         solidity=min(1.0, p.area / max(hull, p.area * 0.5, 1.0)),
                         p_p90=float(np.percentile(prob[ya:yb, xa:xb][sub > 0], 90)), ring_c0=ring_c0,
                         r_cen=float(rr[int(p.centroid[0]), int(p.centroid[1])])))
    df = pd.DataFrame(rows)
    for key, v in f.items():
        df[key] = v
    df["contrast"] = df["ring_c0"] - df["c0_mean"]
    df["n_comp"] = n
    df["area_rank"] = pd.Series(f["area"]).rank(ascending=False).values / n
    if n > 1:
        dm = np.hypot(cents[:, None, 0] - cents[None, :, 0], cents[:, None, 1] - cents[None, :, 1]); np.fill_diagonal(dm, np.inf)
        df["nn_dist"] = dm.min(1) / s
    else:
        df["nn_dist"] = 1e3
    df["img_p_mean"] = float(prob[valid].mean()) if valid.any() else 0.0
    df["station"] = S.STATIONS.index(station) if station in S.STATIONS else -1
    return df[COMP_FEATURES]


def gbm(seed=0):
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=16, min_samples_leaf=20,
                                         l2_regularization=1.0, random_state=seed)


def crossfit(X, y, groups, seed=0, n_splits=5):
    """Out-of-fold predictions (GroupKFold over images: no image scores itself) + a final model on everything."""
    oof = np.zeros(len(y))
    for tr, va in GroupKFold(n_splits=min(n_splits, len(np.unique(groups)))).split(X, y, groups):
        oof[va] = gbm(seed).fit(X[tr], y[tr]).predict(X[va])
    return np.clip(oof, 0, 1), gbm(seed).fit(X, y)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd"); ap.add_argument("--fold", type=int, required=True); ap.add_argument("--hours", type=float, required=True)
    ap.add_argument("--work", required=True); ap.add_argument("--cache", required=True)
    a = ap.parse_args()
    try:
        train_main(a)
    except BaseException:
        import traceback; traceback.print_exc(); sys.exit(1)

In [ ]:
importlib.invalidate_caches()
import sfc_common as S, a_lib as A
importlib.reload(S); importlib.reload(A)


def banner(t):
    print(f"\n{'=' * 90}\n[{(time.time() - T0) / 3600:.2f} h] {t}\n{'=' * 90}", flush=True)

## 1 · Data, physics channels and caches

In [ ]:
banner("DATA + CACHE")
data = S.Data(S.find_root())
stems, R = data.stems, CFG["res"]
print(f"{len(stems)} train images, {sum(len(data.recs[s]) for s in stems)} annotator records, {len(data.test_paths)} test images")
S.build_cache(data, TEMP / f"train_{R}", R)
S.build_cache(data, TEMP / f"test_{R}", R, test=True)
C, TC = S.open_cache(TEMP / f"train_{R}"), S.open_cache(TEMP / f"test_{R}")
folds = S.make_folds(stems, N_FOLDS)
BASE_VAL = S.baseline_val_idx(len(stems))                 # the first U-Net notebook's validation images
json.dump(folds.tolist(), open(WORK / "folds_a.json", "w")); json.dump(CFG, open(WORK / "cfg_a.json", "w"))
print("fold sizes:", np.bincount(folds).tolist(), "| baseline validation images:", len(BASE_VAL))

## 2 · Training: 4 folds, one process per GPU (two folds at a time)

In [ ]:
banner("PRETRAINED ENCODER")
enc_name, enc_src = A.save_pretrained_encoder(CFG, WORK / "enc_init_a.pt", log=print)
if enc_name != CFG["encoder"]:
    CFG["encoder"] = enc_name; json.dump(CFG, open(WORK / "cfg_a.json", "w"))
print("encoder:", CFG["encoder"], "| weights:", enc_src or "NONE")
if not enc_src and not DRY:
    print("!!! no pretrained weights (Internet off?) - the encoder trains from scratch and results will be much weaker")


def run_folds(fold_list, hours):
    slots = list(range(N_GPU)) if N_GPU else [None]
    todo, results = list(fold_list), {}
    for attempt in range(2):
        failed = []
        for w in range(0, len(todo), len(slots)):
            procs = []
            for f, g in zip(todo[w:w + len(slots)], slots):
                env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONPATH=str(WORK))
                if g is not None:
                    env["CUDA_VISIBLE_DEVICES"] = str(g)
                p = subprocess.Popen([sys.executable, "a_lib.py", "train", "--fold", str(f), "--hours", str(hours),
                                      "--work", str(WORK), "--cache", str(TEMP / f"train_{R}")],
                                     cwd=str(WORK), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
                th = threading.Thread(target=lambda p=p, f=f, g=g: [print(f"[fold{f}|gpu{g}] {l.rstrip()}", flush=True) for l in p.stdout])
                th.start(); procs.append((f, p, th))
            for f, p, th in procs:
                p.wait(); th.join()
                ok = p.returncode == 0 and (WORK / f"a_fold{f}.pt").exists()
                if ok:
                    results[f] = json.load(open(WORK / f"a_fold{f}.json"))
                else:
                    print(f"!!! fold {f} failed (code {p.returncode})"); failed.append(f)
        if not failed:
            break
        todo = failed; print("retrying", failed)
    return results


banner("TRAINING")
fold_res = run_folds(list(range(N_FOLDS)), FOLD_HOURS)
print(pd.DataFrame(fold_res.values()).to_string(index=False))
done = sorted(fold_res)
assert done, "no fold trained"

## 3 · Out-of-fold probability maps (each image predicted by the model that never saw it)

In [ ]:
banner("OUT-OF-FOLD INFERENCE")
DEVS = [torch.device(f"cuda:{g}") for g in range(N_GPU)] or [torch.device("cpu")]
OOF = np.lib.format.open_memmap(TEMP / "oof_a.npy", "w+", np.uint8, (len(stems), R, R))


def predict_items(model_paths, cache, items, out=None):
    """Spread images over the GPUs (one thread and one copy of the models per GPU). Returns {item: prob}."""
    chunks = [items[k::len(DEVS)] for k in range(len(DEVS))]

    def work(dev, chunk):
        ms = A.load_models(CFG, model_paths, dev); res = {}
        for i in chunk:
            p = A.predict_prob(ms, cache["chans"][i], cache["valid"][i], dev)
            if out is not None:
                out[i] = np.round(p * 255).astype(np.uint8)
            else:
                res[i] = p
        return res

    with ThreadPoolExecutor(len(DEVS)) as ex:
        parts = list(ex.map(work, DEVS, chunks))
    return {k: v for p in parts for k, v in p.items()}


t = time.time()
for f in done:
    predict_items([str(WORK / f"a_fold{f}.pt")], C, list(np.flatnonzero(folds == f)), OOF)
OOF.flush()
oof_idx = np.flatnonzero(np.isin(folds, done))
print(f"{len(oof_idx)} out-of-fold maps in {time.time() - t:.0f}s")

## 4 · Post-processing tuned on out-of-fold predictions: extraction → fragment merging → component scorer

In [ ]:
banner("POST-PROCESSING TUNING")
NJ = os.cpu_count() or 1
MIN_AREA = int(40 * (R / 1024) ** 2)
prob_of = lambda i: OOF[i].astype(np.float32) / 255
valid_of = lambda cache, i: np.asarray(cache["valid"][i]).astype(bool)
rr_of = lambda cache, i, shape: np.hypot(*np.meshgrid(np.arange(shape[1]) - cache["disk"][i][0], np.arange(shape[0]) - cache["disk"][i][1])) / cache["disk"][i][2]
rec_labels = lambda i: [np.asarray(C["labels"][j]) for j in range(C["rec_start"][i], C["rec_start"][i + 1])]


def gt_res(i):
    """Ground truth at RES from the cached label maps: {record_id: [RLE per filament]}."""
    return {rid: list(S.label_rles(S.relabel(l.astype(np.int32))).values()) for rid, l in zip(data.recs[stems[i]], rec_labels(i))}


GTR = dict(zip(oof_idx, ThreadPoolExecutor(NJ).map(gt_res, oof_idx)))
GT_RES = {rid: v for i in oof_idx for rid, v in GTR[i].items()}


def light_pq(labs, idx, qs=(0.0, 0.3, 0.4, 0.5, 0.6)):
    """Scorer-free yardstick: keep components with mean probability >= q (best q)."""
    enc = {i: S.label_rles(labs[i]) for i in idx}
    pm = {i: {k: float(prob_of(i)[labs[i] == k].mean()) for k in enc[i]} for i in idx}
    gt = {rid: v for i in idx for rid, v in GTR[i].items()}
    best = None
    for q in qs:
        r = S.pq(gt, {stems[i]: [enc[i][k] for k in enc[i] if pm[i][k] >= q] for i in idx})
        if best is None or r["pq"] > best[0]["pq"]:
            best = (r, q)
    return best


rng = np.random.default_rng(0)
SUB = np.sort(rng.permutation(oof_idx)[:min(250, len(oof_idx))])
# (a) extraction thresholds
rows = []
for t_lo, t_hi in [(0.25, 0.45), (0.3, 0.5), (0.35, 0.55), (0.4, 0.6), (0.45, 0.65)]:
    labs = dict(zip(SUB, ThreadPoolExecutor(NJ).map(lambda i: S.instances_cc(prob_of(i), valid_of(C, i), t_hi, t_lo, MIN_AREA), SUB)))
    r, q = light_pq(labs, SUB); rows.append(dict(t_lo=t_lo, t_hi=t_hi, q=q, **r))
ext = pd.DataFrame(rows).sort_values("pq", ascending=False).reset_index(drop=True)
print(ext.to_string(index=False))
T_LO, T_HI = float(ext.t_lo[0]), float(ext.t_hi[0])
LAB0 = dict(zip(oof_idx, ThreadPoolExecutor(NJ).map(lambda i: S.instances_cc(prob_of(i), valid_of(C, i), T_HI, T_LO, MIN_AREA), oof_idx)))

# (b) learned fragment merging
MAX_GAP = 48 * R / 1024


def pairs_of(i):
    pt = A.pair_table(LAB0[i], prob_of(i), C["chans"][i], rr_of(C, i, LAB0[i].shape), MAX_GAP)
    return pt, A.pair_targets(LAB0[i], pt, rec_labels(i))


PT = dict(zip(oof_idx, ThreadPoolExecutor(NJ).map(pairs_of, oof_idx)))
has = [i for i in oof_idx if len(PT[i][0])]
pair_model, pair_oof = None, {}
if has:
    X = np.concatenate([PT[i][0][A.PAIR_FEATURES].to_numpy(float) for i in has]); y = np.concatenate([PT[i][1] for i in has])
    g = np.concatenate([np.full(len(PT[i][0]), i) for i in has])
    oof_s, pair_model = A.crossfit(X, y, g, seed=CFG["seed"])
    k = 0
    for i in has:
        pair_oof[i] = oof_s[k:k + len(PT[i][0])]; k += len(PT[i][0])
    print(f"{len(y)} nearby component pairs | same-filament rate {np.mean(y >= 0.5):.3f}")


def merged(i, thr, scores=None):
    pt = PT[i][0]
    if thr > 1 or not len(pt):
        return LAB0[i]
    sc = pair_oof.get(i) if scores is None else scores
    return A.apply_merges(LAB0[i], pt, sc, thr, valid_of(C, i))


rows = []
for thr in ([0.3, 0.4, 0.5, 0.6, 0.7] if has else []) + [1.01]:
    labs = {i: merged(i, thr) for i in SUB}
    r, q = light_pq(labs, SUB); rows.append(dict(merge_thr=thr, q=q, **r))
mtab = pd.DataFrame(rows).sort_values("pq", ascending=False).reset_index(drop=True)
print(mtab.to_string(index=False), "\n(merge_thr 1.01 = no merging)")
MERGE_THR = float(mtab.merge_thr[0])
LAB = {i: merged(i, MERGE_THR) for i in oof_idx}

# (c) learned component scorer: predicts each component's expected matched IoU; keep it iff that beats the PQ break-even
def comps_of(i):
    tab = A.comp_table(LAB[i], prob_of(i), C["chans"][i], rr_of(C, i, LAB[i].shape), valid_of(C, i), stems[i][-2:])
    return tab, S.matched_iou(LAB[i], GTR[i])


CT = dict(zip(oof_idx, ThreadPoolExecutor(NJ).map(comps_of, oof_idx)))
hasc = [i for i in oof_idx if len(CT[i][0])]
X = np.concatenate([CT[i][0].to_numpy(float) for i in hasc]); y = np.concatenate([CT[i][1] for i in hasc])
g = np.concatenate([np.full(len(CT[i][0]), i) for i in hasc])
u_oof, scorer = A.crossfit(X, y, g, seed=CFG["seed"])
U, k = {}, 0
for i in hasc:
    U[i] = u_oof[k:k + len(CT[i][0])]; k += len(CT[i][0])
ENC = {i: S.label_rles(LAB[i]) for i in oof_idx}


def pq_sel(score_of, thr, idx=oof_idx):
    return S.pq({rid: v for i in idx for rid, v in GTR[i].items()},
                {stems[i]: [ENC[i][k] for k in S.select(score_of(i), thr) if k in ENC[i]] for i in idx})


curve = pd.DataFrame([dict(tau=t, **pq_sel(lambda i: U.get(i, np.zeros(0)), t)) for t in np.round(np.arange(0, 0.62, 0.02), 2)])
TAU = float(curve.tau[curve.pq.idxmax()])
orc = max(pq_sel(lambda i: CT[i][1], t)["pq"] for t in np.round(np.arange(0, 0.62, 0.04), 2))
print(f"{len(y)} components | scorer tau={TAU:.2f}: PQ {curve.pq.max():.4f} (at {R} px) | oracle scorer upper bound {orc:.4f} | "
      f"theory check: tau ~ PQ/2 = {curve.pq.max() / 2:.2f}")

## 5 · Out-of-fold PQ at 2048 px (the leaderboard's resolution), overall and on the baseline's validation images

In [ ]:
banner("OUT-OF-FOLD EVALUATION @2048")
GT_FULL = data.gt(oof_idx, S.FULL)
PRED_FULL = dict(zip([stems[i] for i in oof_idx], ThreadPoolExecutor(NJ).map(
    lambda i: S.to_rles(LAB[i], S.select(U.get(i, np.zeros(0)), TAU), prob_of(i), T_LO, S.FULL), oof_idx)))
sub_gt = lambda idx: {rid: v for rid, v in GT_FULL.items() if rid.split("-", 1)[1] in {stems[i] for i in idx}}
OOF_PQ = S.pq(GT_FULL, PRED_FULL)
BASE_PQ = S.pq(sub_gt([i for i in BASE_VAL if i in set(oof_idx)]), PRED_FULL)
print("out-of-fold PQ, all images        :", OOF_PQ)
print("PQ on the baseline val images      :", BASE_PQ, " <- compare with the first U-Net's validation PQ")
by_st = {st: round(S.pq(sub_gt([i for i in oof_idx if stems[i][-2:] == st]), PRED_FULL)["pq"], 3) for st in S.STATIONS}
print("by station:", by_st)
abl = pd.DataFrame([dict(step="hysteresis + mean-prob rule (no merge, no scorer)", pq=mtab[mtab.merge_thr > 1].pq.iloc[0]),
                    dict(step=f"+ learned merging (thr {MERGE_THR})", pq=mtab.pq.iloc[0]),
                    dict(step=f"+ learned scorer (tau {TAU})", pq=curve.pq.max())])
print("\nablation at", R, "px (250-image subset for the first two rows):\n" + abl.round(4).to_string(index=False))

## 6 · Test: 4-fold ensemble → submission.csv

In [ ]:
banner("TEST INFERENCE")
test_stems = [p.stem for p in data.test_paths]
t = time.time()
TP = predict_items([str(WORK / f"a_fold{f}.pt") for f in done], TC, list(range(len(test_stems))))


def test_instances(i):
    prob, valid = TP[i], valid_of(TC, i)
    rr = rr_of(TC, i, prob.shape)
    lab = S.instances_cc(prob, valid, T_HI, T_LO, MIN_AREA)
    t_lo = T_LO
    if lab.max() == 0 and prob.max() > 0:                  # nothing above threshold: relax to this image's own scale
        t_lo = 0.5 * float(prob.max()); lab = S.instances_cc(prob, valid, 0.8 * float(prob.max()), t_lo, MIN_AREA)
    if pair_model is not None and MERGE_THR <= 1 and lab.max() > 1:
        pt = A.pair_table(lab, prob, TC["chans"][i], rr, MAX_GAP)
        if len(pt):
            lab = A.apply_merges(lab, pt, pair_model.predict(pt[A.PAIR_FEATURES].to_numpy(float)), MERGE_THR, valid)
    if lab.max() == 0:
        return []
    u = np.clip(scorer.predict(A.comp_table(lab, prob, TC["chans"][i], rr, valid, test_stems[i][-2:]).to_numpy(float)), 0, 1)
    return S.to_rles(lab, S.select(u, TAU), prob, t_lo, S.FULL)


preds = dict(zip(test_stems, ThreadPoolExecutor(NJ).map(test_instances, range(len(test_stems)))))
sub = S.write_submission(preds, WORK / "submission.csv")
rep = S.validate_submission(sub, test_stems)
print(rep, f"| {time.time() - t:.0f}s")
if rep["images_without_rows"]:
    print("!!! images without any prediction:", rep["images_without_rows"])
REPORT = dict(approach="A: EfficientNetV2-S U-Net x4 folds @1536 + learned merge + learned scorer", encoder=CFG["encoder"],
              weights=enc_src, oof_pq=OOF_PQ, baseline_val_pq=BASE_PQ, by_station=by_st, t_lo=T_LO, t_hi=T_HI,
              merge_thr=MERGE_THR, tau=TAU, folds=fold_res, submission=rep, hours=round((time.time() - T0) / 3600, 2))
json.dump(REPORT, open(WORK / "report_a.json", "w"), indent=1, default=str)
print(json.dumps(REPORT, indent=1, default=str))
for f in ("enc_init_a.pt",):
    (WORK / f).unlink(missing_ok=True)                     # keep the saved version small